In [3]:
from egomimic.utils.aws.aws_sql import (
    episode_table_to_df,
    create_default_engine,
)

In [4]:
engine = create_default_engine()
df = episode_table_to_df(engine)

Tables in schema 'app': ['episodes']


In [5]:
df

,episode_hash,operator,lab,num_frames,task,task_description,scene,objects,processed_path,mp4_path,is_deleted,embodiment,robot_name,is_eval,eval_score,eval_success,processing_error
0,2025-10-29-22-05-17-825000,Lawrence,rl2,-1,fold clothes,folding clothes,8,"{""short sleeve shirts"",""long sleeve shirts"",sh...",,,False,aria,aria_bimanual,False,-1.0,True,
1,692ea0262fa9ba56c08f8097,6768a79a7fdec9138007a095,mecka,159,object_in_container,,,,rldb:/mecka/flagship/692ea0262fa9ba56c08f8097/,rldb:/mecka/flagship/692ea0262fa9ba56c08f8097_...,False,mecka,mecka_bimanual,False,-1.0,True,
2,2025-10-29-21-54-36-193000,Aniketh,rl2,-1,fold clothes,folding clothes,1,"{""short sleeve shirts"",""long sleeve shirts"",sh...",,,False,aria,aria_bimanual,False,-1.0,True,
3,2025-11-11-22-54-29-457000,rl2,rl2,1400,fold clothes,folding clothes,7,"{""short sleeve shirts"",""long sleeve shirts"",sh...",s3://rldb/processed_v2/aria/2025-11-11-22-54-2...,s3://rldb/processed_v2/aria/1762901669457_vide...,False,aria,aria_bimanual,False,-1.0,True,
4,2025-12-27-21-34-49-000000,Yangcen,rl2,3800,put cup on saucer indomain,"pick up cup with one arm, hand it over to the ...",1,"{""pink cup"",""green square saucer""}",s3://rldb/processed_v2/aria/2025-12-27-21-34-4...,s3://rldb/processed_v2/aria/1766871289000_vide...,False,aria,aria_bimanual,False,-1.0,True,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42730,2026-01-12-01-21-46-914000,Ryan,rl2,675,cup_on_saucer,"pick up cup with one arm, hand it over to the ...",1,"{""white cup"",""green square saucer""}",rldb:/processed_v2/eva/1768180906914_processed,rldb:/processed_v2/eva/1768180906914_video.mp4,False,eva,eva_bimanual,False,-1.0,True,
42731,2026-01-11-18-13-05-672000,Ryan,rl2,1962,cup_on_saucer,"pick up cup with one arm, hand it over to the ...",1,"{""white ceramic cup"",""white saucer""}",rldb:/processed_v2/eva/1768155185672_processed,rldb:/processed_v2/eva/1768155185672_video.mp4,False,eva,eva_bimanual,False,-1.0,True,
42732,2026-01-12-00-45-50-415000,Ryan,rl2,827,cup_on_saucer,"pick up cup with one arm, hand it over to the ...",1,"{""white cup"",""yellow saucer""}",rldb:/processed_v2/eva/1768178750415_processed,rldb:/processed_v2/eva/1768178750415_video.mp4,False,eva,eva_bimanual,False,-1.0,True,
42733,2026-01-11-22-48-36-523000,Ryan,rl2,1361,cup_on_saucer,"pick up cup with one arm, hand it over to the ...",1,"{""white cup"",""white saucer""}",rldb:/processed_v2/eva/1768171716523_processed,rldb:/processed_v2/eva/1768171716523_video.mp4,False,eva,eva_bimanual,False,-1.0,True,


In [6]:
def rename_processed_to_episode_hash(df, dry_run, max_workers=16):
    """
    For each row in the sql table there's processed path of the form rldb:/mecka/flagship/692ea0262fa9ba56c08f8097/	
    We want to change this to s3://rldb/mecka/flagship/<row.episode_hash>/
    There are a lot of rows (42k), ideally the file renaming should be doen in a batch fashion.
    The processed paths are all on S3.
    args:
        df: pandas dataframe of the episode sql table
        dry_run: if True, only print what would be done
        max_workers: number of threads used for S3 copy operations
    """
    import pandas as pd
    import boto3
    from botocore.exceptions import ClientError
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from sqlalchemy import MetaData, Table, bindparam, update
    from egomimic.utils.aws.aws_sql import create_default_engine

    def normalize_processed_path(path):
        if path is None:
            return None
        if isinstance(path, float) and pd.isna(path):
            return None
        text = str(path).strip()
        if not text:
            return None
        if text.startswith("s3://"):
            return text
        if text.startswith("rldb:/"):
            return "s3://rldb/" + text[len("rldb:/"):].lstrip("/")
        if text.startswith("rldb://"):
            return "s3://rldb/" + text[len("rldb://"):].lstrip("/")
        if text.startswith("rldb:"):
            return "s3://rldb/" + text[len("rldb:"):].lstrip("/")
        return None

    def parse_s3_uri(uri):
        if not uri.startswith("s3://"):
            return None, None
        rest = uri[len("s3://"):]
        bucket, _, key = rest.partition("/")
        return bucket, key

    def ensure_trailing_slash(prefix):
        if prefix and not prefix.endswith("/"):
            return prefix + "/"
        return prefix

    def chunked(items, size):
        for i in range(0, len(items), size):
            yield items[i : i + size]

    def move_s3(pairs, chunk_size=100, max_workers=16):
        totals = {"prefixes": 0, "objects": 0, "failures": 0}
        for batch in chunked(pairs, chunk_size):
            for old_uri, new_uri in batch:
                old_bucket, old_prefix = parse_s3_uri(old_uri)
                new_bucket, new_prefix = parse_s3_uri(new_uri)
                if not old_bucket or not new_bucket:
                    totals["failures"] += 1
                    continue

                old_prefix = ensure_trailing_slash(old_prefix.lstrip("/"))
                new_prefix = ensure_trailing_slash(new_prefix.lstrip("/"))

                if dry_run:
                    print(
                        f"[DRY RUN] s3://{old_bucket}/{old_prefix} -> s3://{new_bucket}/{new_prefix}"
                    )

                paginator = s3.get_paginator("list_objects_v2")
                page_iter = paginator.paginate(Bucket=old_bucket, Prefix=old_prefix)
                keys = []
                for page in page_iter:
                    for obj in page.get("Contents", []):
                        keys.append(obj["Key"])

                if not keys:
                    if dry_run:
                        print(f"[DRY RUN] No objects under s3://{old_bucket}/{old_prefix}")
                    continue

                totals["prefixes"] += 1
                totals["objects"] += len(keys)

                if dry_run:
                    continue

                def copy_one(key):
                    new_key = new_prefix + key[len(old_prefix):]
                    try:
                        s3.copy_object(
                            Bucket=new_bucket,
                            Key=new_key,
                            CopySource={"Bucket": old_bucket, "Key": key},
                        )
                        return True, key, new_key, None
                    except ClientError as exc:
                        return False, key, new_key, exc

                to_delete = []
                with ThreadPoolExecutor(max_workers=max_workers) as executor:
                    futures = [executor.submit(copy_one, key) for key in keys]
                    for fut in as_completed(futures):
                        ok, key, new_key, exc = fut.result()
                        if ok:
                            to_delete.append({"Key": key})
                        else:
                            totals["failures"] += 1
                            print(
                                "Failed to copy "
                                f"s3://{old_bucket}/{key} -> s3://{new_bucket}/{new_key}: {exc}"
                            )

                for chunk in chunked(to_delete, 1000):
                    try:
                        s3.delete_objects(Bucket=old_bucket, Delete={"Objects": chunk})
                    except ClientError as exc:
                        totals["failures"] += len(chunk)
                        print(
                            "Failed to delete objects under "
                            f"s3://{old_bucket}/{old_prefix}: {exc}"
                        )

        return totals

    if "processed_path" not in df.columns or "episode_hash" not in df.columns:
        raise ValueError("df must include 'processed_path' and 'episode_hash' columns")

    s3 = boto3.client("s3")
    pairs = []
    updates = []
    skipped = 0

    for _, row in df.iterrows():
        old_uri = normalize_processed_path(row.get("processed_path"))
        episode_hash = row.get("episode_hash")
        if not old_uri or not episode_hash:
            skipped += 1
            continue

        new_uri = f"s3://rldb/mecka/flagship/{episode_hash}/"
        if old_uri.rstrip("/") + "/" == new_uri.rstrip("/") + "/":
            skipped += 1
            continue

        pairs.append((old_uri, new_uri))
        updates.append({"episode_hash": episode_hash, "processed_path": new_uri})

    move_stats = move_s3(pairs, chunk_size=100, max_workers=max_workers)

    if dry_run:
        print(f"[DRY RUN] Would update {len(updates)} rows in app.episodes")
        return {
            "move": move_stats,
            "updates": len(updates),
            "skipped": skipped,
        }

    engine = create_default_engine()
    metadata = MetaData()
    episodes_tbl = Table("episodes", metadata, autoload_with=engine, schema="app")
    stmt = (
        update(episodes_tbl)
        .where(episodes_tbl.c.episode_hash == bindparam("episode_hash"))
        .values(processed_path=bindparam("processed_path"))
    )
    with engine.begin() as conn:
        conn.execute(stmt, updates)

    return {
        "move": move_stats,
        "updates": len(updates),
        "skipped": skipped,
    }


In [7]:
rename_processed_to_episode_hash(df, dry_run=True, max_workers=16)

[DRY RUN] s3://rldb/processed_v2/aria/2025-11-11-22-54-29-457000/ -> s3://rldb/mecka/flagship/2025-11-11-22-54-29-457000/
[DRY RUN] s3://rldb/processed_v2/aria/2025-12-27-21-34-49-000000/ -> s3://rldb/mecka/flagship/2025-12-27-21-34-49-000000/
[DRY RUN] s3://rldb/processed_v2/aria/2025-11-11-22-56-28-321000/ -> s3://rldb/mecka/flagship/2025-11-11-22-56-28-321000/
[DRY RUN] s3://rldb/processed_v2/aria/2025-10-13-03-53-53-876000/ -> s3://rldb/mecka/flagship/2025-10-13-03-53-53-876000/
[DRY RUN] s3://rldb/processed_v2/aria/2025-10-13-03-51-22-623000/ -> s3://rldb/mecka/flagship/2025-10-13-03-51-22-623000/
[DRY RUN] s3://rldb/processed_v2/aria/1758917853000_processed/ -> s3://rldb/mecka/flagship/2025-09-26-20-17-33-000000/
[DRY RUN] s3://rldb/processed_v2/aria/1762901703203_processed/ -> s3://rldb/mecka/flagship/2025-11-11-22-55-03-203000/
[DRY RUN] s3://rldb/processed_v2/aria/1760416233706_processed/ -> s3://rldb/mecka/flagship/2025-10-14-04-30-33-706000/
[DRY RUN] s3://rldb/processed_v2/

KeyboardInterrupt: 